In [1]:
import requests, time
import pandas as pd

int_url = "https://api.carbonintensity.org.uk/intensity/2025-01-15T00:00Z/2025-01-16T00:00Z"
start = time.time()
try:
    r = requests.get(int_url, headers={"Accept": "application/json"}, timeout=(5, 30))
    print("Status:", r.status_code, "| took", round(time.time() - start, 1), "s")
    print(r.text[:300])
except requests.exceptions.ConnectTimeout:
    print("Couldn't connect at all: likely network/firewall")
except requests.exceptions.ReadTimeout:
    print("Connected, but the API is slow to respond")
except Exception as e:
    print("Other error:", type(e).__name__, e)

int_data = r.json()

gen_url = "https://api.carbonintensity.org.uk/generation/2025-01-15T00:00Z/2025-01-16T00:00Z"
try:
    r = requests.get(gen_url)
    print("Status:", r.status_code, "| took", round(time.time() - start, 1), "s")
    print(r.text[:300])
except requests.exceptions.ConnectTimeout:
    print("Couldn't connect at all: likely network/firewall")
except requests.exceptions.ReadTimeout:
    print("Connected, but the API is slow to respond")
except Exception as e:
    print("Other error:", type(e).__name__, e)

gen_data = r.json()

Status: 200 | took 0.2 s
{"data":[{"from":"2025-01-14T23:30Z","to":"2025-01-15T00:00Z","intensity":{"forecast":195,"actual":197,"index":"high"}},{"from":"2025-01-15T00:00Z","to":"2025-01-15T00:30Z","intensity":{"forecast":192,"actual":197,"index":"high"}},{"from":"2025-01-15T00:30Z","to":"2025-01-15T01:00Z","intensity":{"fo
Status: 200 | took 0.4 s
{"data":[{"from":"2025-01-14T23:30Z","to":"2025-01-15T00:00Z","generationmix":[{"fuel":"biomass","perc":7.1},{"fuel":"coal","perc":0},{"fuel":"imports","perc":4.5},{"fuel":"gas","perc":48.3},{"fuel":"nuclear","perc":13.3},{"fuel":"other","perc":0},{"fuel":"hydro","perc":0.2},{"fuel":"solar","perc":0


In [2]:
int_records = int_data["data"]
print(len(int_records))
print(int_records[0])
print(int_records[1])
print(int_records[2])
print(int_records[0].keys())

gen_records = gen_data["data"]
print(len(gen_records))
print(gen_records[0])
print(gen_records[1])
print(gen_records[2])
print(gen_records[0].keys())

49
{'from': '2025-01-14T23:30Z', 'to': '2025-01-15T00:00Z', 'intensity': {'forecast': 195, 'actual': 197, 'index': 'high'}}
{'from': '2025-01-15T00:00Z', 'to': '2025-01-15T00:30Z', 'intensity': {'forecast': 192, 'actual': 197, 'index': 'high'}}
{'from': '2025-01-15T00:30Z', 'to': '2025-01-15T01:00Z', 'intensity': {'forecast': 194, 'actual': 198, 'index': 'high'}}
dict_keys(['from', 'to', 'intensity'])
49
{'from': '2025-01-14T23:30Z', 'to': '2025-01-15T00:00Z', 'generationmix': [{'fuel': 'biomass', 'perc': 7.1}, {'fuel': 'coal', 'perc': 0}, {'fuel': 'imports', 'perc': 4.5}, {'fuel': 'gas', 'perc': 48.3}, {'fuel': 'nuclear', 'perc': 13.3}, {'fuel': 'other', 'perc': 0}, {'fuel': 'hydro', 'perc': 0.2}, {'fuel': 'solar', 'perc': 0}, {'fuel': 'wind', 'perc': 26.6}]}
{'from': '2025-01-15T00:00Z', 'to': '2025-01-15T00:30Z', 'generationmix': [{'fuel': 'biomass', 'perc': 7}, {'fuel': 'coal', 'perc': 0}, {'fuel': 'imports', 'perc': 4.4}, {'fuel': 'gas', 'perc': 48.3}, {'fuel': 'nuclear', 'perc': 

In [5]:
idf = pd.json_normalize(int_records)
idf.head()

,from,to,intensity.forecast,intensity.actual,intensity.index
0,2025-01-14T23:30Z,2025-01-15T00:00Z,195,197,high
1,2025-01-15T00:00Z,2025-01-15T00:30Z,192,197,high
2,2025-01-15T00:30Z,2025-01-15T01:00Z,194,198,high
3,2025-01-15T01:00Z,2025-01-15T01:30Z,197,198,high
4,2025-01-15T01:30Z,2025-01-15T02:00Z,201,197,high


In [4]:
gdf = pd.json_normalize(gen_records)
gdf.head()

,from,to,generationmix
0,2025-01-14T23:30Z,2025-01-15T00:00Z,"[{'fuel': 'biomass', 'perc': 7.1}, {'fuel': 'c..."
1,2025-01-15T00:00Z,2025-01-15T00:30Z,"[{'fuel': 'biomass', 'perc': 7}, {'fuel': 'coa..."
2,2025-01-15T00:30Z,2025-01-15T01:00Z,"[{'fuel': 'biomass', 'perc': 7}, {'fuel': 'coa..."
3,2025-01-15T01:00Z,2025-01-15T01:30Z,"[{'fuel': 'biomass', 'perc': 7.1}, {'fuel': 'c..."
4,2025-01-15T01:30Z,2025-01-15T02:00Z,"[{'fuel': 'biomass', 'perc': 7.2}, {'fuel': 'c..."


In [11]:
test = pd.json_normalize(gen_records, record_path="generationmix", meta=["from", "to"])
test.head()

,fuel,perc,from,to
0,biomass,7.1,2025-01-14T23:30Z,2025-01-15T00:00Z
1,coal,0.0,2025-01-14T23:30Z,2025-01-15T00:00Z
2,imports,4.5,2025-01-14T23:30Z,2025-01-15T00:00Z
3,gas,48.3,2025-01-14T23:30Z,2025-01-15T00:00Z
4,nuclear,13.3,2025-01-14T23:30Z,2025-01-15T00:00Z
